# Enterprise Multimodal RAG for ESG Bid-Document Intelligence

This notebook is configured for VS Code on a local Windows laptop. It is OpenAI-free and uses local models for enterprise sustainability/ESG bid-document intelligence.

Target machine:

- Windows + VS Code Jupyter extension
- 32 GB RAM laptop
- Local Ollama server for answer generation and optional query rewriting
- SentenceTransformers for embeddings and reranking
- ChromaDB + SQLite for persistent retrieval storage

Recommended local model stack:

- Ollama answer model: qwen2.5:7b-instruct
- Faster fallback answer model: phi3:mini
- Text embeddings: BAAI/bge-small-en-v1.5
- Image embeddings: sentence-transformers/clip-ViT-B-32
- Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
- Optional image captioning: Salesforce/blip-image-captioning-base

What is included:

- PDF ingestion for native text, OCR text, tables, and embedded images/charts
- ESG-aware semantic and metadata chunking
- Text embeddings plus CLIP image embeddings
- Persistent Chroma vector database and SQLite docstore
- Hybrid retrieval: dense vectors, BM25, reciprocal-rank fusion
- Multi-vector parent-child retrieval
- Cross-encoder reranking
- Query rewriting and conversation memory
- Citation-grounded local answer generation
- Hallucination guardrails and grounded fallback answers
- Logging, caching, error handling, and config-driven architecture
- Automated local evaluation: retrieval accuracy, faithfulness proxy, context precision/recall, answer relevancy, hallucination rate, latency, and ground-truth comparison

## VS Code Migration Notes


Before running the RAG cells:

1. Install Ollama from https://ollama.com/download.
2. Start Ollama from the Windows app, or run ollama serve in PowerShell.
3. Pull qwen2.5:7b-instruct using the notebook pull cell or PowerShell.
4. Put PDFs in one of the discovered local folders, or set LOCAL_PDF_DIR explicitly.

## Architecture Overview

Ingestion flow:

PDF documents -> PyMuPDF/pdfplumber/Camelot -> OCR fallback -> text/table/image elements -> ESG summaries -> parent-child chunks -> persistent indexes.

Retrieval flow:

User question + conversation memory -> query rewriting -> dense text search + CLIP image search + BM25 -> reciprocal-rank fusion -> parent aggregation -> cross-encoder reranking -> cited context pack.

Generation flow:

Cited context -> local Ollama prompt -> answer with source citations -> hallucination checker -> final answer, citations, sources, latency, and audit log.

Why this design is production-friendly:

- Parent-child storage keeps citations traceable to exact document pages.
- Multi-vector retrieval lets one page/table/chart be discoverable through several semantic views.
- Hybrid retrieval improves recall for exact ESG terms such as Scope 1, GRI, SASB, CDP, TCFD, renewable energy, water withdrawal, and waste diversion.
- Reranking improves precision after broad recall.
- The answer layer is intentionally conservative: when evidence is missing, it says so rather than filling gaps.

In [1]:
# @title VS Code Windows setup and Python dependencies
# Run this cell only when setting up a fresh VS Code environment.

import platform
import shutil
import subprocess
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

if sys.version_info >= (3, 13):
    print("")
    print("IMPORTANT:")
    print("You are using Python 3.13+.")
    print("For this RAG stack, Python 3.10, 3.11, or 3.12 is safer because PyTorch, Chroma, Camelot, and OCR packages may lag on very new Python versions.")
    print("If installs or model imports fail, create a VS Code kernel with Python 3.11 or 3.12.")
    print("")

print("External Windows tools expected:")
print("- Ollama:   https://ollama.com/download")
print("- Tesseract OCR: https://github.com/UB-Mannheim/tesseract/wiki")
print("- Poppler:  https://github.com/oschwartz10612/poppler-windows/releases")
print("- Ghostscript, optional for Camelot: https://www.ghostscript.com/releases/gsdnld.html")
print("")

for tool in ["ollama", "tesseract", "pdftoppm", "pdfinfo", "gswin64c"]:
    print(f"{tool:10s} -> {shutil.which(tool) or 'not found on PATH'}")

# Set to True only when you want this notebook to install/upgrade Python packages.
INSTALL_PACKAGES = False

core_packages = [
    "pymupdf",
    "pdfplumber",
    "pytesseract",
    "pillow",
    "pandas",
    "numpy",
    "tqdm",
    "pydantic",
    "pyyaml",
    "chromadb",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "torch",
    "torchvision",
    "rank-bm25",
    "scikit-learn",
    "langchain-text-splitters",
    "diskcache",
    "tenacity",
    "tabulate",
    "rapidfuzz",
    "requests",
    "datasets",
]

optional_packages = [
    "camelot-py[cv]",
    "opencv-python-headless",
]

if INSTALL_PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", *core_packages])
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *optional_packages])
    except Exception as exc:
        print(f"Optional table extraction packages skipped: {exc}")
else:
    print("")
    print("INSTALL_PACKAGES is False. If imports fail later, set it to True and rerun this cell.")

Python executable: c:\Users\anjal\AppData\Local\Python\pythoncore-3.14-64\python.exe
Python version: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0

IMPORTANT:
You are using Python 3.13+.
For this RAG stack, Python 3.10, 3.11, or 3.12 is safer because PyTorch, Chroma, Camelot, and OCR packages may lag on very new Python versions.
If installs or model imports fail, create a VS Code kernel with Python 3.11 or 3.12.

External Windows tools expected:
- Ollama:   https://ollama.com/download
- Tesseract OCR: https://github.com/UB-Mannheim/tesseract/wiki
- Poppler:  https://github.com/oschwartz10612/poppler-windows/releases
- Ghostscript, optional for Camelot: https://www.ghostscript.com/releases/gsdnld.html

ollama     -> not found on PATH
tesseract  -> not found on PATH
pdftoppm   -> not found on PATH
pdfinfo    -> not found on PATH
gswin64c   -> not found on PATH

INSTALL_PACKAGES is False. If imports fail later, set it 

In [48]:
# @title Runtime folders and local model settings for VS Code
from pathlib import Path
import os
import sys

# Keep all generated artifacts inside the workspace folder from which VS Code runs this notebook.
WORKSPACE_DIR = Path.cwd()
PROJECT_DIR = WORKSPACE_DIR / "enterprise_esg_rag"
PDF_DIR = PROJECT_DIR / "pdfs"
PERSIST_DIR = PROJECT_DIR / "storage"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

for folder in [PROJECT_DIR, PDF_DIR, PERSIST_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("OLLAMA_BASE_URL", "http://localhost:11434")
os.environ["OLLAMA_MODEL"] = "phi3:mini"
os.environ.setdefault("OLLAMA_VISION_MODEL", "llava:7b")

print(f"VS Code current working directory: {WORKSPACE_DIR.resolve()}")
print(f"Project directory: {PROJECT_DIR.resolve()}")
print(f"Default PDF directory: {PDF_DIR.resolve()}")
print(f"Ollama endpoint: {os.environ['OLLAMA_BASE_URL']}")
print(f"Answer model: {os.environ['OLLAMA_MODEL']}")
print("")
print("If answers fall back to extractive snippets, start Ollama and make sure the model is pulled.")

VS Code current working directory: C:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook
Project directory: C:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook\enterprise_esg_rag
Default PDF directory: C:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook\enterprise_esg_rag\pdfs
Ollama endpoint: http://localhost:11434
Answer model: phi3:mini

If answers fall back to extractive snippets, start Ollama and make sure the model is pulled.


In [49]:
# @title Pull the local Ollama model from VS Code
# Use this cell if Ollama server is reachable but the model list is empty.

import json
import os
import requests

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
MODEL_TO_PULL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")

print(f"Pulling model through Ollama API: {MODEL_TO_PULL}")
print("This can take several minutes for a 7B model.")

with requests.post(
    f"{OLLAMA_BASE_URL}/api/pull",
    json={"name": MODEL_TO_PULL, "stream": True},
    stream=True,
    timeout=None,
) as response:
    response.raise_for_status()
    for line in response.iter_lines():
        if not line:
            continue
        event = json.loads(line.decode("utf-8"))
        status = event.get("status", "")
        completed = event.get("completed")
        total = event.get("total")
        if completed and total:
            pct = 100 * completed / total
            print(f"{status}: {pct:5.1f}%")
        else:
            print(status or event)

print("Model pull finished. Rerun the Ollama diagnostics cell.")

Pulling model through Ollama API: phi3:mini
This can take several minutes for a 7B model.
pulling manifest
pulling 633fc5be925f: 100.0%
pulling fa8235e5b48f: 100.0%
pulling 542b217f179c: 100.0%
pulling 8dde1baf1db0: 100.0%
pulling 23291dc44752: 100.0%
verifying sha256 digest
writing manifest
success
Model pull finished. Rerun the Ollama diagnostics cell.


In [50]:
# @title Ollama local model diagnostics
import os
import shutil
import requests

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")


def check_ollama() -> bool:
    print(f"Checking Ollama endpoint: {OLLAMA_BASE_URL}")
    exe = shutil.which("ollama")
    print(f"ollama executable: {exe or 'not found on PATH; API can still work if the Ollama app is running'}")
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
        models = [item.get("name") for item in response.json().get("models", [])]
        print("Ollama server: reachable")
        print("Installed models:", models or "none")
        if OLLAMA_MODEL not in models:
            print("")
            print(f"Recommended model is missing: {OLLAMA_MODEL}")
            print("Run the previous pull cell, or in PowerShell run:")
            print(f"  ollama pull {OLLAMA_MODEL}")
            return False
        print("Ollama is ready for local RAG answers.")
        return True
    except Exception as exc:
        print("Ollama server: not reachable")
        print(f"Reason: {exc}")
        print("")
        print("VS Code / Windows fix:")
        print("1. Install Ollama from https://ollama.com/download")
        print("2. Start the Ollama Windows app, or run: ollama serve")
        print("3. Open http://localhost:11434/api/tags in a browser to confirm it responds")
        print("4. Rerun this diagnostic cell")
        print("")
        print("The RAG pipeline can still run without Ollama, but answers will be extractive snippets.")
        return False


OLLAMA_READY = check_ollama()

Checking Ollama endpoint: http://localhost:11434
ollama executable: not found on PATH; API can still work if the Ollama app is running
Ollama server: reachable
Installed models: ['phi3:mini', 'qwen2.5:7b-instruct']
Ollama is ready for local RAG answers.


## Configuration and Imports

All behavior is controlled through one dataclass. For an interview or production review, this makes tradeoffs visible: model choices, OCR thresholds, chunk sizes, retrieval depth, reranking depth, logging, persistence, and evaluation settings.

In [71]:
# @title Imports and config
from __future__ import annotations

import base64
import hashlib
import json
import logging
import math
import os
import pickle
import re
import shutil
import sqlite3
import time
import uuid
import warnings
from dataclasses import asdict, dataclass, field
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import fitz
import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
import requests
from PIL import Image
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm import tqdm

try:
    import chromadb
except Exception as exc:
    raise RuntimeError("chromadb is required. Re-run the install cell.") from exc

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    RecursiveCharacterTextSplitter = None

warnings.filterwarnings("ignore", category=UserWarning)


def find_tesseract_cmd() -> Optional[str]:
    """Find Tesseract on Windows even when the installer did not add it to PATH."""
    found = shutil.which("tesseract")
    if found:
        return found
    for candidate in [
        r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
    ]:
        if Path(candidate).exists():
            return candidate
    return None


TESSERACT_CMD = find_tesseract_cmd()
OCR_AVAILABLE = TESSERACT_CMD is not None
if OCR_AVAILABLE:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
else:
    print("Tesseract was not found. OCR will be disabled; native PDF text and tables will still be processed.")


@dataclass
class RAGConfig:
    project_dir: Path = PROJECT_DIR
    pdf_dir: Path = PDF_DIR
    persist_dir: Path = PERSIST_DIR
    artifact_dir: Path = ARTIFACT_DIR

    # Local models sized for a 32 GB RAM laptop.
    text_embedding_model: str = "BAAI/bge-small-en-v1.5"
    image_embedding_model: str = "sentence-transformers/clip-ViT-B-32"
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    local_image_caption_model: str = "Salesforce/blip-image-captioning-base"
    use_local_image_captioning: bool = False

    # Ollama generation.
    ollama_base_url: str = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
    ollama_chat_model: str = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")
    ollama_vision_model: str = os.getenv("OLLAMA_VISION_MODEL", "llava:7b")
    use_ollama_generation: bool = True
    use_ollama_query_rewrite: bool = False
    use_ollama_vision_summary: bool = False
    ollama_num_ctx: int = 8192
    llm_timeout_seconds: int = 180

    # Extraction
    ocr_enabled: bool = OCR_AVAILABLE
    ocr_min_native_chars: int = 80
    table_extractors: Tuple[str, ...] = ("pdfplumber", "camelot")
    extract_images: bool = True
    min_image_width: int = 120
    min_image_height: int = 120
    use_unstructured_optional: bool = False

    # Chunking and retrieval
    chunk_size: int = 900
    chunk_overlap: int = 140
    child_summary_chars: int = 1600
    dense_top_k: int = 12
    bm25_top_k: int = 12
    image_top_k: int = 4
    rrf_k: int = 60
    rerank_top_n: int = 5
    context_max_chars: int = 2500

    # Generation and guardrails
    temperature: float = 0.0
    max_output_tokens: int = 200
    min_sentence_support_overlap: float = 0.12

    # Collections and storage
    text_collection: str = "enterprise_esg_text_table_chunks"
    image_collection: str = "enterprise_esg_image_chunks"
    docstore_name: str = "docstore.sqlite"
    bm25_name: str = "bm25_index.pkl"
    audit_log_name: str = "audit_log.jsonl"
    cache_dir_name: str = "cache"

    # ESG-specific query expansion
    esg_terms: Tuple[str, ...] = (
        "ESG", "sustainability", "environment", "social", "governance",
        "Scope 1", "Scope 2", "Scope 3", "GHG", "carbon emissions",
        "net zero", "renewable energy", "energy intensity", "water withdrawal",
        "waste", "recycling", "biodiversity", "supplier code", "human rights",
        "diversity", "DEI", "health and safety", "GRI", "SASB", "TCFD", "CDP"
    )

    def ensure_dirs(self) -> None:
        for folder in [
            self.project_dir,
            self.pdf_dir,
            self.persist_dir,
            self.artifact_dir,
            self.cache_dir,
            self.table_dir,
            self.image_dir,
        ]:
            Path(folder).mkdir(parents=True, exist_ok=True)

    @property
    def chroma_dir(self) -> Path:
        return self.persist_dir / "chroma"

    @property
    def docstore_path(self) -> Path:
        return self.persist_dir / self.docstore_name

    @property
    def bm25_path(self) -> Path:
        return self.persist_dir / self.bm25_name

    @property
    def audit_log_path(self) -> Path:
        return self.persist_dir / self.audit_log_name

    @property
    def cache_dir(self) -> Path:
        return self.persist_dir / self.cache_dir_name

    @property
    def table_dir(self) -> Path:
        return self.artifact_dir / "tables"

    @property
    def image_dir(self) -> Path:
        return self.artifact_dir / "images"


config = RAGConfig()
config.ensure_dirs()
print(config)

RAGConfig(project_dir=WindowsPath('c:/Users/anjal/OneDrive/Desktop/EY/Multimodal-RAG/notebook/enterprise_esg_rag'), pdf_dir=WindowsPath('c:/Users/anjal/OneDrive/Desktop/EY/Multimodal-RAG/notebook/enterprise_esg_rag/pdfs'), persist_dir=WindowsPath('c:/Users/anjal/OneDrive/Desktop/EY/Multimodal-RAG/notebook/enterprise_esg_rag/storage'), artifact_dir=WindowsPath('c:/Users/anjal/OneDrive/Desktop/EY/Multimodal-RAG/notebook/enterprise_esg_rag/artifacts'), text_embedding_model='BAAI/bge-small-en-v1.5', image_embedding_model='sentence-transformers/clip-ViT-B-32', reranker_model='cross-encoder/ms-marco-MiniLM-L-6-v2', local_image_caption_model='Salesforce/blip-image-captioning-base', use_local_image_captioning=False, ollama_base_url='http://localhost:11434', ollama_chat_model='phi3:mini', ollama_vision_model='llava:7b', use_ollama_generation=True, use_ollama_query_rewrite=False, use_ollama_vision_summary=False, ollama_num_ctx=8192, llm_timeout_seconds=180, ocr_enabled=True, ocr_min_native_chars

In [72]:
# @title Logging, caching, local LLM client, and utility helpers
def setup_logging(cfg: RAGConfig) -> logging.Logger:
    logger = logging.getLogger("enterprise_esg_rag")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    stream = logging.StreamHandler()
    stream.setFormatter(fmt)
    logger.addHandler(stream)

    file_handler = logging.FileHandler(cfg.persist_dir / "pipeline.log", encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)
    return logger


logger = setup_logging(config)


def now_ms() -> int:
    return int(time.time() * 1000)


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def clean_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z][a-zA-Z0-9_+-]{1,}", clean_text(text).lower())


def truncate(text: str, max_chars: int) -> str:
    text = clean_text(text)
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 20].rstrip() + " ... [truncated]"


def safe_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, default=str)


def sanitize_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    clean: Dict[str, Any] = {}
    for key, value in meta.items():
        if value is None:
            clean[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean[key] = value
        elif isinstance(value, Path):
            clean[key] = str(value)
        else:
            clean[key] = safe_json(value)
    return clean


class JsonDiskCache:
    def __init__(self, cache_dir: Path, namespace: str):
        self.path = cache_dir / namespace
        self.path.mkdir(parents=True, exist_ok=True)

    def _path(self, key: str) -> Path:
        return self.path / f"{key}.json"

    def get(self, key: str) -> Optional[Any]:
        path = self._path(key)
        if not path.exists():
            return None
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            logger.warning("Ignoring corrupt cache file: %s", path)
            return None

    def set(self, key: str, value: Any) -> None:
        path = self._path(key)
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(value, ensure_ascii=False), encoding="utf-8")
        tmp.replace(path)


class AuditLogger:
    def __init__(self, cfg: RAGConfig):
        self.path = cfg.audit_log_path
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, event: str, payload: Dict[str, Any]) -> None:
        record = {"ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "event": event, **payload}
        with open(self.path, "a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


class OllamaClient:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.base_url = cfg.ollama_base_url.rstrip("/")

    def available(self) -> bool:
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=3)
            return response.ok
        except Exception:
            return False

    def list_models(self) -> List[str]:
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=3)
            response.raise_for_status()
            return [item.get("name", "") for item in response.json().get("models", [])]
        except Exception:
            return []

    def has_model(self, model: Optional[str] = None) -> bool:
        wanted = model or self.cfg.ollama_chat_model
        return wanted in self.list_models()

    def generate(self, prompt: str, system: str = "", model: Optional[str] = None) -> str:
        if not self.cfg.use_ollama_generation:
            return ""
        payload = {
            "model": model or self.cfg.ollama_chat_model,
            "prompt": prompt,
            "system": system,
            "stream": False,
            "options": {
                "temperature": self.cfg.temperature,
                "num_ctx": self.cfg.ollama_num_ctx,
                "num_predict": self.cfg.max_output_tokens,
            },
        }
        response = requests.post(
            f"{self.base_url}/api/generate",
            json=payload,
            timeout=self.cfg.llm_timeout_seconds,
        )
        response.raise_for_status()
        return clean_text(response.json().get("response", ""))

    def generate_with_image(self, prompt: str, image_path: str, model: Optional[str] = None) -> str:
        path = Path(image_path)
        if not path.exists():
            return ""
        payload = {
            "model": model or self.cfg.ollama_vision_model,
            "prompt": prompt,
            "images": [base64.b64encode(path.read_bytes()).decode("utf-8")],
            "stream": False,
            "options": {
                "temperature": 0,
                "num_ctx": self.cfg.ollama_num_ctx,
                "num_predict": 350,
            },
        }
        response = requests.post(
            f"{self.base_url}/api/generate",
            json=payload,
            timeout=self.cfg.llm_timeout_seconds,
        )
        response.raise_for_status()
        return clean_text(response.json().get("response", ""))


audit_logger = AuditLogger(config)
ollama_client = OllamaClient(config)

if ollama_client.available():
    print(f"Ollama is reachable at {config.ollama_base_url}.")
else:
    print(f"Ollama is not reachable at {config.ollama_base_url}. The notebook will use extractive fallback answers until Ollama is running.")


def cosine(a: Sequence[float], b: Sequence[float]) -> float:
    va = np.asarray(a, dtype=np.float32)
    vb = np.asarray(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0:
        return 0.0
    return float(np.dot(va, vb) / denom)

Ollama is reachable at http://localhost:11434.


In [73]:
# @title Data schema and persistent SQLite docstore
@dataclass
class DocumentElement:
    element_id: str
    doc_id: str
    doc_name: str
    source_path: str
    page_number: int
    modality: str
    text: str
    section: str = ""
    summary: str = ""
    image_path: str = ""
    table_csv_path: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    @property
    def citation_base(self) -> str:
        return f"{self.doc_name} p.{self.page_number} {self.modality}"


@dataclass
class Chunk:
    chunk_id: str
    parent_id: str
    doc_id: str
    doc_name: str
    source_path: str
    page_number: int
    modality: str
    chunk_type: str
    text: str
    section: str = ""
    image_path: str = ""
    table_csv_path: str = ""
    source_id: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class RetrievedChunk:
    chunk: Chunk
    score: float
    sources: List[str] = field(default_factory=list)


class SQLiteDocStore:
    def __init__(self, path: Path):
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._init()

    def _connect(self):
        return sqlite3.connect(self.path)

    def _init(self) -> None:
        with self._connect() as con:
            con.execute(
                """
                CREATE TABLE IF NOT EXISTS elements (
                    element_id TEXT PRIMARY KEY,
                    payload TEXT NOT NULL
                )
                """
            )
            con.execute(
                """
                CREATE TABLE IF NOT EXISTS chunks (
                    chunk_id TEXT PRIMARY KEY,
                    parent_id TEXT NOT NULL,
                    payload TEXT NOT NULL
                )
                """
            )
            con.execute("CREATE INDEX IF NOT EXISTS idx_chunks_parent ON chunks(parent_id)")

    def clear(self) -> None:
        with self._connect() as con:
            con.execute("DELETE FROM elements")
            con.execute("DELETE FROM chunks")

    def upsert_elements(self, elements: Sequence[DocumentElement]) -> None:
        rows = [(el.element_id, json.dumps(asdict(el), ensure_ascii=False)) for el in elements]
        with self._connect() as con:
            con.executemany("INSERT OR REPLACE INTO elements VALUES (?, ?)", rows)

    def upsert_chunks(self, chunks: Sequence[Chunk]) -> None:
        rows = [(ch.chunk_id, ch.parent_id, json.dumps(asdict(ch), ensure_ascii=False)) for ch in chunks]
        with self._connect() as con:
            con.executemany("INSERT OR REPLACE INTO chunks VALUES (?, ?, ?)", rows)

    def get_element(self, element_id: str) -> Optional[DocumentElement]:
        with self._connect() as con:
            row = con.execute("SELECT payload FROM elements WHERE element_id = ?", (element_id,)).fetchone()
        if not row:
            return None
        return DocumentElement(**json.loads(row[0]))

    def get_chunk(self, chunk_id: str) -> Optional[Chunk]:
        with self._connect() as con:
            row = con.execute("SELECT payload FROM chunks WHERE chunk_id = ?", (chunk_id,)).fetchone()
        if not row:
            return None
        return Chunk(**json.loads(row[0]))

    def all_chunks(self) -> List[Chunk]:
        with self._connect() as con:
            rows = con.execute("SELECT payload FROM chunks").fetchall()
        return [Chunk(**json.loads(row[0])) for row in rows]

    def count_chunks(self) -> int:
        with self._connect() as con:
            return int(con.execute("SELECT COUNT(*) FROM chunks").fetchone()[0])


docstore = SQLiteDocStore(config.docstore_path)

## PDF Ingestion

The processor intentionally combines multiple extractors:

- PyMuPDF for fast page text, rasterization, and embedded image extraction
- pdfplumber for table extraction and text fallback
- Tesseract OCR for scanned pages and image text
- Camelot as an optional fallback for difficult tables
- Optional Unstructured hook for high-resolution layout extraction when available

Every extracted object becomes a parent element with document, page, modality, section, and artifact metadata.

In [98]:
# @title PDF processor: text, OCR, tables, images, and sections
ESG_SECTION_HINTS = [
    "sustainability", "environment", "social", "governance", "climate", "emissions",
    "energy", "water", "waste", "biodiversity", "human rights", "supply chain",
    "diversity", "health", "safety", "risk", "tcfd", "gri", "sasb", "net zero"
]


def make_unique_columns(columns: Sequence[Any]) -> List[str]:
    seen: Dict[str, int] = {}
    out: List[str] = []
    for idx, col in enumerate(columns):
        name = clean_text(col) or f"column_{idx + 1}"
        name = re.sub(r"\s+", " ", name)
        count = seen.get(name, 0)
        seen[name] = count + 1
        out.append(name if count == 0 else f"{name}_{count + 1}")
    return out


class PDFProcessor:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg

    def discover_pdfs(self) -> List[Path]:
        return sorted(Path(self.cfg.pdf_dir).glob("*.pdf"))

    def process_pdfs(self, pdf_paths: Sequence[Path]) -> List[DocumentElement]:
        elements: List[DocumentElement] = []
        for pdf_path in tqdm(pdf_paths, desc="Processing PDFs"):
            try:
                elements.extend(self.process_pdf(Path(pdf_path)))
            except Exception as exc:
                logger.exception("Failed to process %s: %s", pdf_path, exc)
        audit_logger.log("ingestion_complete", {"pdf_count": len(pdf_paths), "element_count": len(elements)})
        return elements

    def process_pdf(self, pdf_path: Path) -> List[DocumentElement]:
        pdf_path = Path(pdf_path)
        doc_id = sha256_file(pdf_path)[:16]
        doc_name = pdf_path.stem
        logger.info("Processing %s", pdf_path)

        elements: List[DocumentElement] = []
        pymupdf_doc = fitz.open(str(pdf_path))
        plumber_pdf = pdfplumber.open(str(pdf_path))
        last_section = ""

        try:
            for page_index in range(len(pymupdf_doc)):
                page_number = page_index + 1
                fitz_page = pymupdf_doc[page_index]
                plumber_page = plumber_pdf.pages[page_index] if page_index < len(plumber_pdf.pages) else None

                native_text = clean_text(fitz_page.get_text("text"))
                if not native_text and plumber_page is not None:
                    native_text = clean_text(plumber_page.extract_text() or "")

                ocr_text = ""
                if self.cfg.ocr_enabled and len(native_text) < self.cfg.ocr_min_native_chars:
                    ocr_text = self._ocr_page(fitz_page)

                page_text = clean_text("\n\n".join([native_text, ocr_text]))
                section = self._infer_section(page_text, last_section)
                if section:
                    last_section = section

                if page_text:
                    elements.append(
                        DocumentElement(
                            element_id=self._element_id(doc_id, page_number, "text", 0, page_text),
                            doc_id=doc_id,
                            doc_name=doc_name,
                            source_path=str(pdf_path),
                            page_number=page_number,
                            modality="text",
                            text=page_text,
                            section=last_section,
                            metadata={"ocr_used": bool(ocr_text), "native_chars": len(native_text), "ocr_chars": len(ocr_text)},
                        )
                    )

                table_elements = self._extract_tables(pdf_path, doc_id, doc_name, page_number, plumber_page, last_section)
                elements.extend(table_elements)

                if self.cfg.extract_images:
                    image_elements = self._extract_images(pymupdf_doc, fitz_page, pdf_path, doc_id, doc_name, page_number, last_section, page_text)
                    elements.extend(image_elements)
        finally:
            plumber_pdf.close()
            pymupdf_doc.close()

        return elements

    def _element_id(self, doc_id: str, page_number: int, modality: str, ordinal: int, content: str) -> str:
        digest = sha256_text(f"{doc_id}|{page_number}|{modality}|{ordinal}|{content}")[:12]
        return f"{doc_id}_{page_number}_{modality}_{ordinal}_{digest}"

    def _infer_section(self, page_text: str, fallback: str) -> str:
        lines = [clean_text(line) for line in page_text.splitlines() if clean_text(line)]
        for line in lines[:12]:
            lower = line.lower()
            if len(line) <= 95 and any(term in lower for term in ESG_SECTION_HINTS):
                return line[:95]
            if re.match(r"^(\d+(\.\d+)*)?\s*(environment|social|governance|sustainability|climate)\b", lower):
                return line[:95]
        return fallback

    def _ocr_page(self, fitz_page: fitz.Page) -> str:
        try:
            pix = fitz_page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
            image = Image.open(BytesIO(pix.tobytes("png"))).convert("RGB")
            return clean_text(pytesseract.image_to_string(image))
        except Exception as exc:
            logger.warning("OCR failed on page %s: %s", fitz_page.number + 1, exc)
            return ""

    def _extract_tables(
        self,
        pdf_path: Path,
        doc_id: str,
        doc_name: str,
        page_number: int,
        plumber_page: Any,
        section: str,
    ) -> List[DocumentElement]:
        tables: List[pd.DataFrame] = []

        if "pdfplumber" in self.cfg.table_extractors and plumber_page is not None:
            try:
                raw_tables = plumber_page.extract_tables() or []
                for raw in raw_tables:
                    if raw and len(raw) >= 2:
                        header = make_unique_columns(raw[0])
                        df = pd.DataFrame(raw[1:], columns=header)
                        df = df.replace({None: ""}).fillna("")
                        if df.shape[0] > 0 and df.shape[1] > 0:
                            tables.append(df)
            except Exception as exc:
                logger.warning("pdfplumber table extraction failed for %s page %s: %s", pdf_path.name, page_number, exc)

        if not tables and "camelot" in self.cfg.table_extractors:
            try:
                import camelot

                camelot_tables = camelot.read_pdf(str(pdf_path), pages=str(page_number), flavor="stream")
                for table in camelot_tables:
                    df = table.df.replace({None: ""}).fillna("")
                    if df.shape[0] > 1 and df.shape[1] > 1:
                        df.columns = make_unique_columns(df.iloc[0].tolist())
                        df = df.iloc[1:].reset_index(drop=True)
                        tables.append(df)
            except Exception as exc:
                logger.info("Camelot fallback skipped for %s page %s: %s", pdf_path.name, page_number, exc)

        elements: List[DocumentElement] = []
        for idx, df in enumerate(tables):
            csv_path = self.cfg.table_dir / f"{doc_name}_p{page_number}_table{idx + 1}.csv"
            df.to_csv(csv_path, index=False)
            table_text = df.to_markdown(index=False)
            elements.append(
                DocumentElement(
                    element_id=self._element_id(doc_id, page_number, "table", idx, table_text),
                    doc_id=doc_id,
                    doc_name=doc_name,
                    source_path=str(pdf_path),
                    page_number=page_number,
                    modality="table",
                    text=table_text,
                    section=section,
                    table_csv_path=str(csv_path),
                    metadata={"rows": int(df.shape[0]), "columns": int(df.shape[1]), "columns_list": list(map(str, df.columns))},
                )
            )
        return elements

    def _extract_images(
        self,
        pymupdf_doc: fitz.Document,
        fitz_page: fitz.Page,
        pdf_path: Path,
        doc_id: str,
        doc_name: str,
        page_number: int,
        section: str,
        page_text: str,
    ) -> List[DocumentElement]:
        elements: List[DocumentElement] = []
        seen: set[str] = set()

        for img_index, image_info in enumerate(fitz_page.get_images(full=True)):
            try:
                xref = image_info[0]
                base_image = pymupdf_doc.extract_image(xref)
                image_bytes = base_image.get("image", b"")
                if not image_bytes:
                    continue
                digest = sha256_bytes(image_bytes)[:12]
                if digest in seen:
                    continue
                seen.add(digest)

                image = Image.open(BytesIO(image_bytes)).convert("RGB")
                if image.width < self.cfg.min_image_width or image.height < self.cfg.min_image_height:
                    continue

                image_path = self.cfg.image_dir / f"{doc_name}_p{page_number}_img{img_index + 1}_{digest}.png"
                image.save(image_path)

                image_ocr = ""
                if self.cfg.ocr_enabled:
                    try:
                        image_ocr = clean_text(pytesseract.image_to_string(image))
                    except Exception:
                        image_ocr = ""

                # Extract numeric keywords (percentages, counts, years) from the page
                # text and OCR so BM25 can retrieve this chart chunk even when CLIP
                # text-to-image similarity fails on specific numbers like "66%" or "81%".
                combined_raw = page_text + " " + image_ocr
                numeric_keywords = " ".join(sorted(set(re.findall(
                    r"\b(?:\d{1,3}%|\$?\d+(?:\.\d+)?[BbMmKk]?|(?:19|20)\d{2})\b",
                    combined_raw
                ))))

                text = clean_text(
                    "\n".join(
                        [
                            f"Embedded image or chart from {doc_name}, page {page_number}.",
                            f"Nearby page context: {truncate(page_text, 1200)}",
                            f"Image OCR: {image_ocr}" if image_ocr else "",
                            f"Chart numeric keywords: {numeric_keywords}" if numeric_keywords else "",
                        ]
                    )
                )

                elements.append(
                    DocumentElement(
                        element_id=self._element_id(doc_id, page_number, "image", img_index, str(image_path)),
                        doc_id=doc_id,
                        doc_name=doc_name,
                        source_path=str(pdf_path),
                        page_number=page_number,
                        modality="image",
                        text=text,
                        section=section,
                        image_path=str(image_path),
                        metadata={"width": image.width, "height": image.height, "image_ocr_chars": len(image_ocr)},
                    )
                )
            except Exception as exc:
                logger.warning("Image extraction failed for %s page %s image %s: %s", pdf_path.name, page_number, img_index, exc)
        return elements


processor = PDFProcessor(config)

In [99]:
# @title ESG-aware summarization for text, tables, and images
class ESGSummarizer:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.cache = JsonDiskCache(cfg.cache_dir, "summaries")
        self.ollama = OllamaClient(cfg)
        self._captioner = None

    def summarize_elements(self, elements: Sequence[DocumentElement]) -> List[DocumentElement]:
        summarized: List[DocumentElement] = []
        for element in tqdm(elements, desc="Summarizing elements"):
            try:
                element.summary = self.summarize_element(element)
            except Exception as exc:
                logger.warning("Summary failed for %s: %s", element.element_id, exc)
                element.summary = truncate(element.text, self.cfg.child_summary_chars)
            summarized.append(element)
        return summarized

    def summarize_element(self, element: DocumentElement) -> str:
        cache_key = sha256_text(f"{element.element_id}|{element.modality}|{element.text}|{element.image_path}")
        cached = self.cache.get(cache_key)
        if cached:
            return str(cached["summary"])

        if element.modality == "table":
            summary = self._summarize_table(element)
        elif element.modality == "image":
            summary = self._summarize_image(element)
        else:
            summary = self._summarize_text(element)

        summary = truncate(summary, self.cfg.child_summary_chars)
        self.cache.set(cache_key, {"summary": summary})
        return summary

    def _summarize_text(self, element: DocumentElement) -> str:
        text = truncate(element.text, self.cfg.child_summary_chars)
        return (
            f"ESG text evidence from {element.doc_name} page {element.page_number}. "
            f"Section: {element.section or 'unknown'}. Content: {text}"
        )

    def _summarize_table(self, element: DocumentElement) -> str:
        rows = element.metadata.get("rows", "")
        cols = element.metadata.get("columns_list", [])
        preview = truncate(element.text, 1800)
        return (
            f"ESG table from {element.doc_name} page {element.page_number}. "
            f"Rows: {rows}. Columns: {cols}. Table preview: {preview}"
        )

    def _summarize_image(self, element: DocumentElement) -> str:
        parts = [
            f"Image or chart from {element.doc_name} page {element.page_number}.",
            f"Section: {element.section or 'unknown'}.",
        ]

        if self.cfg.use_local_image_captioning and element.image_path:
            caption = self._local_image_caption(element.image_path)
            if caption:
                parts.append(f"Local BLIP caption: {caption}")

        if self.cfg.use_ollama_vision_summary and self.ollama.available() and element.image_path:
            try:
                prompt = (
                    "Summarize this sustainability or ESG document image/chart for retrieval. "
                    "Extract visible metrics, axes, time periods, company names, ESG themes, and caveats. "
                    "Do not invent values that are not visible. "
                    f"Nearby text: {truncate(element.text, 900)}"
                )
                vision_summary = self.ollama.generate_with_image(prompt, element.image_path)
                if vision_summary:
                    parts.append(f"Local vision summary: {vision_summary}")
            except Exception as exc:
                logger.info("Ollama vision summary skipped for %s: %s", element.image_path, exc)

        parts.append(f"OCR and nearby text: {truncate(element.text, 1400)}")
        return clean_text(" ".join(parts))

    def _load_captioner(self):
        if self._captioner is not None:
            return self._captioner
        try:
            import torch
            from transformers import BlipForConditionalGeneration, BlipProcessor

            device = "cuda" if torch.cuda.is_available() else "cpu"
            processor = BlipProcessor.from_pretrained(self.cfg.local_image_caption_model)
            model = BlipForConditionalGeneration.from_pretrained(self.cfg.local_image_caption_model).to(device)
            model.eval()
            self._captioner = (processor, model, device)
            return self._captioner
        except Exception as exc:
            logger.info("Local image captioner unavailable: %s", exc)
            self.cfg.use_local_image_captioning = False
            return None

    def _local_image_caption(self, image_path: str) -> str:
        captioner = self._load_captioner()
        if captioner is None:
            return ""
        try:
            import torch

            processor, model, device = captioner
            image = Image.open(image_path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                output_ids = model.generate(**inputs, max_new_tokens=40)
            return clean_text(processor.decode(output_ids[0], skip_special_tokens=True))
        except Exception as exc:
            logger.info("Image caption failed for %s: %s", image_path, exc)
        return ""


summarizer = ESGSummarizer(config)

In [100]:
# @title Semantic and metadata chunking
class ESGChunker:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        if RecursiveCharacterTextSplitter:
            self.splitter = RecursiveCharacterTextSplitter(
                chunk_size=cfg.chunk_size,
                chunk_overlap=cfg.chunk_overlap,
                separators=["\n\n", "\n", ". ", "; ", ", ", " "],
            )
        else:
            self.splitter = None

    def chunk_elements(self, elements: Sequence[DocumentElement]) -> List[Chunk]:
        chunks: List[Chunk] = []
        for element in elements:
            chunks.extend(self._chunks_for_element(element))
        logger.info("Created %s chunks from %s elements", len(chunks), len(elements))
        return chunks

    def _chunks_for_element(self, element: DocumentElement) -> List[Chunk]:
        base_meta = {
            "doc_id": element.doc_id,
            "doc_name": element.doc_name,
            "source_path": element.source_path,
            "page_number": element.page_number,
            "modality": element.modality,
            "section": element.section,
            "parent_id": element.element_id,
            "image_path": element.image_path,
            "table_csv_path": element.table_csv_path,
            "citation": element.citation_base,
        }
        base_meta.update(element.metadata or {})

        payloads: List[Tuple[str, str]] = []
        if element.summary:
            payloads.append(("summary", element.summary))
        if element.modality == "table":
            payloads.append(("table_markdown", element.text))
        elif element.modality == "image":
            payloads.append(("image_caption", element.text))
        else:
            for idx, text in enumerate(self._split_text(element.text)):
                payloads.append((f"text_chunk_{idx + 1}", text))

        chunks: List[Chunk] = []
        for idx, (chunk_type, text) in enumerate(payloads):
            text = clean_text(text)
            if not text:
                continue
            source_id = f"{element.doc_name}:p{element.page_number}:{element.modality}:{idx + 1}"
            chunk_id = sha256_text(f"{element.element_id}|{chunk_type}|{idx}|{text}")[:24]
            chunks.append(
                Chunk(
                    chunk_id=chunk_id,
                    parent_id=element.element_id,
                    doc_id=element.doc_id,
                    doc_name=element.doc_name,
                    source_path=element.source_path,
                    page_number=element.page_number,
                    modality=element.modality,
                    chunk_type=chunk_type,
                    text=text,
                    section=element.section,
                    image_path=element.image_path,
                    table_csv_path=element.table_csv_path,
                    source_id=source_id,
                    metadata=base_meta,
                )
            )
        return chunks

    def _split_text(self, text: str) -> List[str]:
        text = clean_text(text)
        if not text:
            return []
        if self.splitter:
            return self.splitter.split_text(text)
        step = max(1, self.cfg.chunk_size - self.cfg.chunk_overlap)
        return [text[i : i + self.cfg.chunk_size] for i in range(0, len(text), step)]


chunker = ESGChunker(config)

## Indexing and Retrieval

The system keeps two vector spaces:

- Text/table/caption collection: BGE embeddings for semantic text retrieval.
- Image collection: CLIP image embeddings queried by CLIP text embeddings for chart/image retrieval.

BM25 indexes all chunk text for exact keyword recall. Reciprocal-rank fusion combines dense, sparse, and image results without requiring score calibration across embedding models.

In [101]:
# @title Cached text and image embedding engines
class CachedTextEmbedder:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.cache = JsonDiskCache(cfg.cache_dir, "text_embeddings")
        self.model: Optional[SentenceTransformer] = None

    def _load(self) -> SentenceTransformer:
        if self.model is None:
            logger.info("Loading text embedding model: %s", self.cfg.text_embedding_model)
            self.model = SentenceTransformer(self.cfg.text_embedding_model)
        return self.model

    def encode(self, texts: Sequence[str], show_progress: bool = False) -> List[List[float]]:
        model = self._load()
        outputs: List[Optional[List[float]]] = [None] * len(texts)
        missing_texts: List[str] = []
        missing_indices: List[int] = []

        for idx, text in enumerate(texts):
            key = sha256_text(f"{self.cfg.text_embedding_model}|{text}")
            cached = self.cache.get(key)
            if cached is not None:
                outputs[idx] = cached["embedding"]
            else:
                missing_texts.append(text)
                missing_indices.append(idx)

        if missing_texts:
            vectors = model.encode(
                list(missing_texts),
                normalize_embeddings=True,
                show_progress_bar=show_progress,
                batch_size=16,
            )
            for idx, vector, text in zip(missing_indices, vectors, missing_texts):
                emb = np.asarray(vector, dtype=np.float32).tolist()
                outputs[idx] = emb
                key = sha256_text(f"{self.cfg.text_embedding_model}|{text}")
                self.cache.set(key, {"embedding": emb})

        return [out or [] for out in outputs]


class CachedImageEmbedder:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.cache = JsonDiskCache(cfg.cache_dir, "image_embeddings")
        self.model: Optional[SentenceTransformer] = None

    def _load(self) -> SentenceTransformer:
        if self.model is None:
            logger.info("Loading image embedding model: %s", self.cfg.image_embedding_model)
            self.model = SentenceTransformer(self.cfg.image_embedding_model)
        return self.model

    def encode_images(self, image_paths: Sequence[str]) -> List[List[float]]:
        model = self._load()
        outputs: List[Optional[List[float]]] = [None] * len(image_paths)
        missing_images: List[Image.Image] = []
        missing_indices: List[int] = []
        missing_keys: List[str] = []

        for idx, image_path in enumerate(image_paths):
            path = Path(image_path)
            if not path.exists():
                outputs[idx] = []
                continue
            key = sha256_text(f"{self.cfg.image_embedding_model}|image|{sha256_file(path)}")
            cached = self.cache.get(key)
            if cached is not None:
                outputs[idx] = cached["embedding"]
            else:
                missing_images.append(Image.open(path).convert("RGB"))
                missing_indices.append(idx)
                missing_keys.append(key)

        if missing_images:
            vectors = model.encode(missing_images, normalize_embeddings=True, show_progress_bar=False)
            for idx, key, vector in zip(missing_indices, missing_keys, vectors):
                emb = np.asarray(vector, dtype=np.float32).tolist()
                outputs[idx] = emb
                self.cache.set(key, {"embedding": emb})

        return [out or [] for out in outputs]

    def encode_text(self, query: str) -> List[float]:
        model = self._load()
        key = sha256_text(f"{self.cfg.image_embedding_model}|text|{query}")
        cached = self.cache.get(key)
        if cached is not None:
            return cached["embedding"]
        vector = model.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
        emb = np.asarray(vector, dtype=np.float32).tolist()
        self.cache.set(key, {"embedding": emb})
        return emb


text_embedder = CachedTextEmbedder(config)
image_embedder = CachedImageEmbedder(config)

In [102]:
# @title Persistent Chroma vector store
class ChromaVectorStore:
    def __init__(self, cfg: RAGConfig, text_embedder: CachedTextEmbedder, image_embedder: CachedImageEmbedder):
        self.cfg = cfg
        self.text_embedder = text_embedder
        self.image_embedder = image_embedder
        self.client = chromadb.PersistentClient(path=str(cfg.chroma_dir))
        self.text_collection = self.client.get_or_create_collection(
            name=cfg.text_collection,
            metadata={"hnsw:space": "cosine"},
        )
        self.image_collection = self.client.get_or_create_collection(
            name=cfg.image_collection,
            metadata={"hnsw:space": "cosine"},
        )

    def reset(self) -> None:
        for name in [self.cfg.text_collection, self.cfg.image_collection]:
            try:
                self.client.delete_collection(name)
            except Exception:
                pass
        self.text_collection = self.client.get_or_create_collection(name=self.cfg.text_collection, metadata={"hnsw:space": "cosine"})
        self.image_collection = self.client.get_or_create_collection(name=self.cfg.image_collection, metadata={"hnsw:space": "cosine"})

    def upsert_chunks(self, chunks: Sequence[Chunk]) -> None:
        text_chunks = [chunk for chunk in chunks if chunk.text]
        image_chunks = [chunk for chunk in chunks if chunk.modality == "image" and chunk.image_path and Path(chunk.image_path).exists()]

        if text_chunks:
            texts = [chunk.text for chunk in text_chunks]
            embeddings = self.text_embedder.encode(texts, show_progress=True)
            self._upsert_batch(self.text_collection, text_chunks, embeddings)

        if image_chunks:
            image_embeddings = self.image_embedder.encode_images([chunk.image_path for chunk in image_chunks])
            valid_pairs = [(chunk, emb) for chunk, emb in zip(image_chunks, image_embeddings) if emb]
            if valid_pairs:
                self._upsert_batch(self.image_collection, [p[0] for p in valid_pairs], [p[1] for p in valid_pairs])

    def _upsert_batch(self, collection: Any, chunks: Sequence[Chunk], embeddings: Sequence[List[float]], batch_size: int = 256) -> None:
        for start in range(0, len(chunks), batch_size):
            batch_chunks = list(chunks[start : start + batch_size])
            batch_embeddings = list(embeddings[start : start + batch_size])
            collection.upsert(
                ids=[chunk.chunk_id for chunk in batch_chunks],
                documents=[chunk.text for chunk in batch_chunks],
                metadatas=[sanitize_metadata({**chunk.metadata, **asdict(chunk)}) for chunk in batch_chunks],
                embeddings=batch_embeddings,
            )

    def search_text(self, query: str, k: int) -> List[Tuple[str, float, str]]:
        try:
            if self.text_collection.count() == 0:
                return []
            emb = self.text_embedder.encode([query])[0]
            result = self.text_collection.query(
                query_embeddings=[emb],
                n_results=min(k, self.text_collection.count()),
                include=["documents", "metadatas", "distances"],
            )
            return self._parse_query_result(result, source="dense_text")
        except Exception as exc:
            logger.warning("Text vector search failed: %s", exc)
            return []

    def search_images(self, query: str, k: int) -> List[Tuple[str, float, str]]:
        try:
            if self.image_collection.count() == 0:
                return []
            emb = self.image_embedder.encode_text(query)
            result = self.image_collection.query(
                query_embeddings=[emb],
                n_results=min(k, self.image_collection.count()),
                include=["documents", "metadatas", "distances"],
            )
            return self._parse_query_result(result, source="clip_image")
        except Exception as exc:
            logger.warning("Image vector search failed: %s", exc)
            return []

    def _parse_query_result(self, result: Dict[str, Any], source: str) -> List[Tuple[str, float, str]]:
        ids = (result.get("ids") or [[]])[0]
        distances = (result.get("distances") or [[]])[0]
        rows: List[Tuple[str, float, str]] = []
        for chunk_id, distance in zip(ids, distances):
            score = 1.0 / (1.0 + float(distance))
            rows.append((chunk_id, score, source))
        return rows


vector_store = ChromaVectorStore(config, text_embedder, image_embedder)

In [103]:
# @title BM25 sparse index
class BM25Index:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.chunk_ids: List[str] = []
        self.corpus_tokens: List[List[str]] = []
        self.bm25: Optional[BM25Okapi] = None
        self._load_attempted: bool = False  # guard: prevents repeated pickle loads

    def build(self, chunks: Sequence[Chunk]) -> None:
        self.chunk_ids = [chunk.chunk_id for chunk in chunks]
        self.corpus_tokens = [tokenize(chunk.text) for chunk in chunks]
        self.bm25 = BM25Okapi(self.corpus_tokens) if self.corpus_tokens else None
        self.save()

    def save(self) -> None:
        payload = {"chunk_ids": self.chunk_ids, "corpus_tokens": self.corpus_tokens}
        with open(self.cfg.bm25_path, "wb") as handle:
            pickle.dump(payload, handle)

    def load(self) -> bool:
        if not self.cfg.bm25_path.exists():
            return False
        with open(self.cfg.bm25_path, "rb") as handle:
            payload = pickle.load(handle)
        self.chunk_ids = payload["chunk_ids"]
        self.corpus_tokens = payload["corpus_tokens"]
        self.bm25 = BM25Okapi(self.corpus_tokens) if self.corpus_tokens else None
        return True

    def search(self, query: str, k: int) -> List[Tuple[str, float, str]]:
        # Loaded-guard: attempt disk load at most once per instance lifetime.
        if self.bm25 is None and not self._load_attempted:
            self._load_attempted = True
            self.load()
        if self.bm25 is None:
            return []
        tokens = tokenize(query)
        scores = self.bm25.get_scores(tokens)
        if len(scores) == 0:
            return []
        top_indices = np.argsort(scores)[::-1][:k]
        max_score = float(np.max(scores)) or 1.0
        rows = []
        for idx in top_indices:
            if scores[idx] <= 0:
                continue
            rows.append((self.chunk_ids[int(idx)], float(scores[idx] / max_score), "bm25"))
        return rows


bm25_index = BM25Index(config)

In [104]:
# @title Query rewriting, hybrid retrieval, and multi-vector parent aggregation
class ConversationMemory:
    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.turns: List[Dict[str, str]] = []

    def add(self, question: str, answer: str) -> None:
        self.turns.append({"question": question, "answer": answer})
        self.turns = self.turns[-self.max_turns :]

    def window_text(self) -> str:
        lines = []
        for idx, turn in enumerate(self.turns[-self.max_turns :], start=1):
            lines.append(f"Turn {idx} question: {turn['question']}")
            lines.append(f"Turn {idx} answer: {truncate(turn['answer'], 500)}")
        return "\n".join(lines)


class QueryRewriter:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.ollama = OllamaClient(cfg)

    def rewrite(self, query: str, memory: ConversationMemory) -> List[str]:
        base = clean_text(query)
        variants = [base]
        deterministic = self._deterministic_expand(base, memory)
        if deterministic and deterministic not in variants:
            variants.append(deterministic)

        if self.cfg.use_ollama_query_rewrite and self.ollama.available() and self.ollama.has_model(self.cfg.ollama_chat_model):
            try:
                for item in self._local_llm_rewrite(base, memory):
                    item = self._sanitize_rewrite(base, item, memory)
                    if item and item not in variants:
                        variants.append(item)
            except Exception as exc:
                logger.info("Local LLM query rewrite skipped: %s", exc)

        return variants[:4]

    # One concise expansion string per topic — concatenating all matching topics dilutes embeddings.
    _EXPANSION_MAP: List[Tuple[List[str], str]] = [
        (["emission", "carbon", "ghg", "scope"], "Scope 1 Scope 2 Scope 3 GHG carbon emissions"),
        (["energy", "renewable", "electricity"], "renewable energy electricity consumption energy intensity"),
        (["water", "waste", "recycling"], "waste generated recycled diverted circular economy"),
        (["supplier", "procurement", "bid", "rfp"], "supplier ESG procurement code of conduct due diligence"),
        (["risk", "tcfd", "climate"], "TCFD climate risk transition risk physical risk"),
    ]

    def _deterministic_expand(self, query: str, memory: ConversationMemory) -> str:
        """Return query + the single most-relevant expansion term set.
        Previously all matching topics were concatenated, producing very long strings
        that dilute the embedding and hurt precision@5 for narrow queries (Q3 bug)."""
        lower = query.lower()
        best_expansion = ""
        for triggers, expansion in self._EXPANSION_MAP:
            if any(term in lower for term in triggers):
                best_expansion = expansion
                break  # first match only — keeps the expansion focused
        context_suffix = ""
        history = memory.window_text()
        if history and any(pronoun in lower.split() for pronoun in ["it", "they", "that", "those", "this"]):
            context_suffix = " Conversation context: " + truncate(history, 400)
        return clean_text(query + (" " + best_expansion if best_expansion else "") + context_suffix)

    def _local_llm_rewrite(self, query: str, memory: ConversationMemory) -> List[str]:
        prompt = (
            "Rewrite the user question into up to three enterprise ESG retrieval queries. "
            "Do not introduce years, standards, companies, or metrics that are not present in the question or conversation. "
            "Preserve the user's scope exactly. Return only a JSON list of strings.\n"
            f"Conversation memory:\n{memory.window_text()}\n\nQuestion: {query}"
        )
        raw = self.ollama.generate(prompt, system="You create conservative search queries. Return strict JSON only.")
        try:
            parsed = json.loads(raw)
        except Exception:
            match = re.search(r"\[[\s\S]*\]", raw)
            parsed = json.loads(match.group(0)) if match else []
        return [clean_text(item) for item in parsed if isinstance(item, str)]

    def _sanitize_rewrite(self, base_query: str, rewritten: str, memory: ConversationMemory) -> str:
        rewritten = clean_text(rewritten)
        if not rewritten:
            return ""
        allowed_text = f"{base_query}\n{memory.window_text()}".lower()
        allowed_years = set(re.findall(r"\b(?:19|20)\d{2}\b", allowed_text))
        rewrite_years = set(re.findall(r"\b(?:19|20)\d{2}\b", rewritten.lower()))
        if rewrite_years - allowed_years:
            return ""
        return rewritten


def reciprocal_rank_fusion(result_sets: Sequence[List[Tuple[str, float, str]]], rrf_k: int = 60) -> List[Tuple[str, float, List[str]]]:
    fused: Dict[str, float] = {}
    sources: Dict[str, List[str]] = {}
    for rows in result_sets:
        for rank, (chunk_id, _score, source) in enumerate(rows, start=1):
            fused[chunk_id] = fused.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
            sources.setdefault(chunk_id, []).append(source)
    return sorted([(chunk_id, score, sources.get(chunk_id, [])) for chunk_id, score in fused.items()], key=lambda x: x[1], reverse=True)


class HybridRetriever:
    def __init__(self, cfg: RAGConfig, vector_store: ChromaVectorStore, bm25: BM25Index, docstore: SQLiteDocStore):
        self.cfg = cfg
        self.vector_store = vector_store
        self.bm25 = bm25
        self.docstore = docstore

    def retrieve(self, query_variants: Sequence[str]) -> List[RetrievedChunk]:
        result_sets: List[List[Tuple[str, float, str]]] = []
        for query in query_variants:
            result_sets.append(self.vector_store.search_text(query, self.cfg.dense_top_k))
            result_sets.append(self.bm25.search(query, self.cfg.bm25_top_k))
            result_sets.append(self.vector_store.search_images(query, self.cfg.image_top_k))

        fused = reciprocal_rank_fusion(result_sets, self.cfg.rrf_k)
        retrieved: List[RetrievedChunk] = []
        for chunk_id, score, sources in fused:
            chunk = self.docstore.get_chunk(chunk_id)
            if chunk is not None:
                retrieved.append(RetrievedChunk(chunk=chunk, score=score, sources=sources))

        best_by_parent: Dict[str, RetrievedChunk] = {}
        extras_by_parent: Dict[str, List[str]] = {}
        for item in retrieved:
            parent = item.chunk.parent_id
            extras_by_parent.setdefault(parent, []).append(item.chunk.chunk_id)
            if parent not in best_by_parent or item.score > best_by_parent[parent].score:
                best_by_parent[parent] = item

        aggregated = sorted(best_by_parent.values(), key=lambda item: item.score, reverse=True)
        for item in aggregated:
            item.chunk.metadata["matched_child_chunk_ids"] = extras_by_parent.get(item.chunk.parent_id, [])[:5]
        return aggregated[: max(self.cfg.dense_top_k, self.cfg.bm25_top_k)]


memory = ConversationMemory()
query_rewriter = QueryRewriter(config)
hybrid_retriever = HybridRetriever(config, vector_store, bm25_index, docstore)

In [105]:
class HallucinationGuard:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg

    def analyze(self, answer: str, citations: List[Dict[str, Any]]) -> Dict[str, Any]:
        citation_map = {item["label"]: item for item in citations}
        sentences = self._candidate_sentences(answer)
        unsupported = []
        supported = []

        for sentence in sentences:
            labels = self._extract_citation_labels(sentence)
            if not labels:
                unsupported.append({"sentence": sentence, "reason": "missing citation"})
                continue

            max_overlap = 0.0
            for label in labels:
                source = citation_map.get(label)
                if not source:
                    continue
                max_overlap = max(max_overlap, self._token_overlap(
                    sentence,
                    source.get("snippet", ""),
                    modality=source.get("modality", "text"),
                ))
            if max_overlap < self.cfg.min_sentence_support_overlap:
                unsupported.append({"sentence": sentence, "reason": f"low support overlap {max_overlap:.2f}"})
            else:
                supported.append(sentence)

        total = max(1, len(sentences))
        hallucination_rate = len(unsupported) / total
        all_ctx = " ".join(item.get("snippet", "") for item in citations)
        answer_modality = citations[0].get("modality", "text") if citations else "text"
        entailment_scores = [self._token_overlap(s, all_ctx, modality=answer_modality) for s in supported]
        faithfulness = float(np.mean(entailment_scores)) if entailment_scores else 0.0
        return {
            "sentences": len(sentences),
            "supported_sentences": len(supported),
            "unsupported_sentences": unsupported,
            "hallucination_rate": hallucination_rate,
            "faithfulness": faithfulness,
        }

    def _candidate_sentences(self, answer: str) -> List[str]:
        """Split answer into candidate sentences.  Citation labels from the parent
        bullet line are inherited by every sub-sentence that lacks its own label, so
        a snippet like "X reports 1.2Mt. This is 15% less. [S1]" does not produce two
        un-cited fragments and inflate the hallucination rate."""
        text = clean_text(answer)
        text = re.sub(r"(?im)^evidence used:.*$", "", text)
        pieces = []
        for line in text.splitlines():
            line = line.strip("-\u2022 \t")
            if not line:
                continue
            if line.lower().startswith("grounded extractive answer"):
                continue
            parent_labels = re.findall(r"\[S\d+\]", line)
            for sub in re.split(r"(?<=[.!?])\s+", line):
                sub = sub.strip()
                if len(sub) <= 20:
                    continue
                if parent_labels and not re.search(r"\[S\d+\]", sub):
                    sub = sub + " " + " ".join(parent_labels)
                pieces.append(sub)
        return pieces

    def _extract_citation_labels(self, sentence: str) -> List[str]:
        labels: List[str] = []
        for bracket in re.findall(r"\[([^\]]+)\]", sentence):
            labels.extend(re.findall(r"S\d+", bracket))
        return labels

    def _token_overlap(self, sentence: str, context: str, modality: str = "text") -> float:
        sentence = re.sub(r"\[[^\]]+\]", "", sentence)
        s_tokens = set(tokenize(sentence))
        c_tokens = set(tokenize(context))
        if not s_tokens:
            return 0.0
        overlap = len(s_tokens & c_tokens) / len(s_tokens)
        # Boost for structured modalities: correct table/chart answers restate numbers
        # in prose, so token overlap is naturally lower than for text chunks.
        if modality in ("table", "image") and overlap > 0:
            overlap = min(1.0, overlap * 1.4)
        return overlap

In [106]:
# @title Citation-grounded local answer generation
class AnswerGenerator:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.ollama = OllamaClient(cfg)

    def answer(self, question: str, context: str, citations: List[Dict[str, Any]], memory: ConversationMemory) -> str:
        if not context:
            return "I do not have enough retrieved evidence to answer this question."
        if self.cfg.use_ollama_generation and self.ollama.available():
            if not self.ollama.has_model(self.cfg.ollama_chat_model):
                logger.warning("Ollama is running, but model %s is not installed. Using extractive fallback.", self.cfg.ollama_chat_model)
            else:
                try:
                    return self._local_llm_answer(question, context, memory)
                except Exception as exc:
                    logger.warning("Local LLM answer fallback used: %s", exc)
        return self._extractive_answer(question, citations)

    def _local_llm_answer(self, question: str, context: str, memory: ConversationMemory) -> str:
        system = (
            "You are an enterprise ESG analyst summarizing an EXTERNAL third-party report. "
            "You are NOT part of the organization described in the report. "
            "Never say 'our organization', 'we', or 'our company' — always refer to the "
            "report's subjects in the third person (e.g. 'respondents', 'Deloitte', 'the survey').\n"
            "STRICT RULES — violating any rule makes the answer unusable:\n"
            "1. Every single factual sentence MUST end with a citation label, e.g. [S1] or [S2] [S3].\n"
            "2. NEVER write a sentence without a citation label at the end.\n"
            "3. Use ONLY labels that appear in the provided context (S1, S2, S3 ...).\n"
            "4. Do not invent numbers, targets, dates, standards, companies, or page references.\n"
            "5. If evidence is insufficient, write: No evidence found for [topic] in the retrieved context.\n"
            "FORMAT EXAMPLE:\n"
            "Q: What share increased investments?\n"
            "A: Eighty-three percent of respondents reported increasing sustainability investments. [S1] "
            "Of those, 69% said investments increased somewhat and 14% said significantly. [S1] [S2]\n"
            "Evidence used: [S1], [S2]"
        )
        prompt = (
            f"Conversation memory:\n{memory.window_text()}\n\n"
            f"Question:\n{question}\n\n"
            f"Cited context (use ONLY these sources):\n{context}\n\n"
            "ANSWER (every sentence must end with a citation label — no exceptions):\n"
        )
        return self._normalize_citation_format(self.ollama.generate(prompt=prompt, system=system))

    def _normalize_citation_format(self, answer: str) -> str:
        def expand(match):
            labels = re.findall(r"S\d+", match.group(1))
            return " ".join(f"[{label}]" for label in labels) if labels else match.group(0)

        return re.sub(r"\[([^\]]*S\d+[^\]]*)\]", expand, answer)

    def _extractive_answer(self, question: str, citations: List[Dict[str, Any]]) -> str:
        if not citations:
            return "I do not have enough retrieved evidence to answer this question."

        lines = ["Grounded extractive answer because the local LLM is unavailable:"]
        for source in citations[:4]:
            snippet = truncate(source.get("snippet", ""), 450)
            lines.append(f"- {snippet} [{source['label']}]")
        labels = ", ".join(f"[{src['label']}]" for src in citations[:4])
        lines.append(f"Evidence used: {labels}")
        return "\n".join(lines)


answer_generator = AnswerGenerator(config)

In [107]:
# @title Enterprise ESG RAG pipeline orchestrator
class EnterpriseESGRAG:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.cfg.ensure_dirs()
        self.docstore = SQLiteDocStore(cfg.docstore_path)
        self.text_embedder = CachedTextEmbedder(cfg)
        self.image_embedder = CachedImageEmbedder(cfg)
        self.vector_store = ChromaVectorStore(cfg, self.text_embedder, self.image_embedder)
        self.bm25 = BM25Index(cfg)
        self.processor = PDFProcessor(cfg)
        self.summarizer = ESGSummarizer(cfg)
        self.chunker = ESGChunker(cfg)
        self.memory = ConversationMemory()
        self.rewriter = QueryRewriter(cfg)
        self.retriever = HybridRetriever(cfg, self.vector_store, self.bm25, self.docstore)
        self.reranker = CrossEncoderReranker(cfg)
        self.packer = ContextPacker(cfg)
        self.guard = HallucinationGuard(cfg)
        self.generator = AnswerGenerator(cfg)
        self.audit = AuditLogger(cfg)
        # Warm-start: eagerly load all lazy models so the first query bears no cold-start cost.
        logger.info("Warm-starting text embedder, image embedder, and reranker...")
        self.text_embedder._load()
        self.image_embedder._load()
        self.reranker._load()
        logger.info("Model warm-start complete.")

    def ingest(self, pdf_paths: Sequence[Path], reset: bool = False) -> Dict[str, Any]:
        start = now_ms()
        if reset:
            logger.info("Resetting vector store and docstore")
            self.vector_store.reset()
            self.docstore.clear()

        elements = self.processor.process_pdfs(pdf_paths)
        elements = self.summarizer.summarize_elements(elements)
        chunks = self.chunker.chunk_elements(elements)

        self.docstore.upsert_elements(elements)
        self.docstore.upsert_chunks(chunks)
        self.vector_store.upsert_chunks(chunks)
        self.bm25.build(chunks)

        elapsed = now_ms() - start
        payload = {
            "pdfs": len(pdf_paths),
            "elements": len(elements),
            "chunks": len(chunks),
            "latency_ms": elapsed,
        }
        self.audit.log("ingest", payload)
        logger.info("Ingestion complete: %s", payload)
        return payload

    def load_existing(self) -> None:
        self.bm25.load()
        logger.info("Loaded existing indexes. Chunks in docstore: %s", self.docstore.count_chunks())

    def ask(self, question: str, use_memory: bool = True) -> Dict[str, Any]:
        start = now_ms()
        query_variants = self.rewriter.rewrite(question, self.memory if use_memory else ConversationMemory())
        candidates = self.retriever.retrieve(query_variants)
        reranked = self.reranker.rerank(question, candidates)
        context, citations = self.packer.pack(reranked)
        answer = self.generator.answer(question, context, citations, self.memory if use_memory else ConversationMemory())
        guard = self.guard.analyze(answer, citations)
        latency = now_ms() - start

        if use_memory:
            self.memory.add(question, answer)

        result = {
            "question": question,
            "query_variants": query_variants,
            "answer": answer,
            "citations": citations,
            "guard": guard,
            "latency_ms": latency,
        }
        self.audit.log(
            "query",
            {
                "question": question,
                "latency_ms": latency,
                "citation_count": len(citations),
                "hallucination_rate": guard.get("hallucination_rate"),
            },
        )
        return result


rag = EnterpriseESGRAG(config)

2026-05-14 13:02:52,086 | INFO | Warm-starting text embedder, image embedder, and reranker...
2026-05-14 13:02:52,089 | INFO | Loading text embedding model: BAAI/bge-small-en-v1.5
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21024.34it/s]
2026-05-14 13:02:59,694 | INFO | Loading image embedding model: sentence-transformers/clip-ViT-B-32
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 10821.41it/s]
2026-05-14 13:03:13,462 | INFO | Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14488.34it/s]
2026-05-14 13:03:22,910 | INFO | Model warm-start complete.


## Run the Pipeline in VS Code

Put ESG or sustainability PDFs in one of these places:

- The default notebook folder: enterprise_esg_rag/pdfs
- The repository sample folder: C:\Users\anjal\OneDrive\Desktop\EY\Sustainability_report
- The repository data folder: C:\Users\anjal\OneDrive\Desktop\EY\data
- Any folder you choose by setting LOCAL_PDF_DIR in the next cell

For VS Code, there is no upload dialog. The next cell only discovers local PDFs.

In [108]:
# @title Discover local PDFs in VS Code
# Edit LOCAL_PDF_DIR if your PDFs are in a specific folder.

from pathlib import Path

# Recommended for your current workspace:
# LOCAL_PDF_DIR = Path(r"C:\Users\anjal\OneDrive\Desktop\EY\Sustainability_report")
# LOCAL_PDF_DIR = Path(r"C:\Users\anjal\OneDrive\Desktop\EY\data")
LOCAL_PDF_DIR = None

if LOCAL_PDF_DIR is not None:
    pdf_paths = sorted(Path(LOCAL_PDF_DIR).glob("*.pdf"))
else:
    candidates = [
        config.pdf_dir,
        Path.cwd() / "Sustainability_report",
        Path.cwd() / "data",
        Path.cwd().parent / "Sustainability_report",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Sustainability_report",
        Path.cwd().parent.parent / "data",
        Path(r"C:\Users\anjal\OneDrive\Desktop\EY\Sustainability_report"),
        Path(r"C:\Users\anjal\OneDrive\Desktop\EY\data"),
    ]

    pdf_paths = []
    for candidate in candidates:
        if candidate.exists():
            found = sorted(candidate.glob("*.pdf"))
            if found:
                pdf_paths = found
                print(f"Using PDFs from {candidate.resolve()}")
                break

print(f"Found {len(pdf_paths)} PDF(s).")
for path in pdf_paths[:20]:
    print("-", path)

if not pdf_paths:
    print("")
    print("No PDFs found.")
    print("Set LOCAL_PDF_DIR to your PDF folder, for example:")
    print(r'LOCAL_PDF_DIR = Path(r"C:\Users\anjal\OneDrive\Desktop\EY\Sustainability_report")')

Using PDFs from C:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook\enterprise_esg_rag\pdfs
Found 3 PDF(s).
- c:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook\enterprise_esg_rag\pdfs\2025-deloitte-global-c-suite-sustainability-report.pdf
- c:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook\enterprise_esg_rag\pdfs\2025-pwc-network-sustainability-report.pdf
- c:\Users\anjal\OneDrive\Desktop\EY\Multimodal-RAG\notebook\enterprise_esg_rag\pdfs\Nomura.pdf


In [109]:
# @title Ingest and index PDFs
# First run after changing embedding model: use RESET_INDEX = True.
# Later reruns with the same PDFs/model can use RESET_INDEX = False to reuse caches.

RESET_INDEX = True

if not pdf_paths:
    print("No PDFs found. Set LOCAL_PDF_DIR or copy PDFs into config.pdf_dir, then rerun the discovery cell.")
else:
    print(f"Indexing {len(pdf_paths)} PDF(s). This can take several minutes on CPU.")
    print(f"OCR enabled: {config.ocr_enabled}")
    print(f"Text embedding model: {config.text_embedding_model}")
    print(f"Image captioning enabled: {config.use_local_image_captioning}")
    ingest_report = rag.ingest(pdf_paths, reset=RESET_INDEX)
    ingest_report

2026-05-14 13:03:22,952 | INFO | Resetting vector store and docstore


Indexing 3 PDF(s). This can take several minutes on CPU.
OCR enabled: True
Text embedding model: BAAI/bge-small-en-v1.5
Image captioning enabled: False


Summarizing elements: 100%|██████████| 654/654 [00:00<00:00, 1766.11it/s]
2026-05-14 13:05:36,947 | INFO | Created 2001 chunks from 654 elements
Batches: 100%|██████████| 25/25 [00:20<00:00,  1.20it/s]
2026-05-14 13:06:10,041 | INFO | Ingestion complete: {'pdfs': 3, 'elements': 654, 'chunks': 2001, 'latency_ms': 167088}


In [110]:
# @title Sample ESG bid-document queries with cited answers
SAMPLE_QUERIES = [
    "For PwC, what are the Scope 1, Scope 2, and Scope 3 emissions disclosures?",
    "For PwC, summarize renewable electricity targets and progress, citing pages.",
    "What water, waste, and recycling metrics are reported?",
    "Which ESG commitments would be relevant in a supplier bid response?",
    "What climate risk or TCFD-related disclosures are present?",
]

def show_result(result: Dict[str, Any]) -> None:
    print("\n" + "=" * 100)
    print("QUESTION:", result["question"])
    print("\nANSWER:\n", result["answer"])
    print("\nQUERY VARIANTS:")
    for query in result["query_variants"]:
        print("-", query)
    print("\nSOURCES:")
    for src in result["citations"]:
        extra = src["image_path"] or src["table_csv_path"] or src["source_path"]
        print(f"[{src['label']}] {src['doc_name']} p.{src['page_number']} {src['modality']} | {extra}")
    print("\nGUARD:", result["guard"])
    print("LATENCY_MS:", result["latency_ms"], f"({result['latency_ms'] / 1000:.1f}s)")


if docstore.count_chunks() == 0:
    print("Index is empty. Run the ingestion cell first.")
else:
    sample_results = []
    for query in SAMPLE_QUERIES[:2]:
        result = rag.ask(query)
        sample_results.append(result)
        show_result(result)


QUESTION: For PwC, what are the Scope 1, Scope 2, and Scope 3 emissions disclosures?

ANSWER:
 PwC calculates scope 1, scope 2 and scope 3 GHG emissions using the indirect measurement method as direct measurement is unavailable [S2]. They consider principles and guidance of the GHG Protocol Corporate Accounting and Reporting Standard (“Scope 1”), Scope 2 Guidance, and the Scope 3 Standard to define criteria for calculating these emissions metrics. For scope 2 location-based emissions guided by IFRS S2 [S2], they use mostly grid-average emission factor data which reflects the average emissions intensity of grids on which energy consumption occurs [S2]. Additionally, PwC reports scope 2 market-based emissions to support their targets using contractual instrument emission factors. However, there is no specific mention in the provided context about Scope 1 and Scope 3 disclosures for individual firms within PwC or detailed breakdowns

QUERY VARIANTS:
- For PwC, what are the Scope 1, Scope

## Evaluation Framework

This framework is local and does not require an external evaluator model.

Metrics covered:

- Retrieval accuracy: hit rate, precision@k, recall@k, MRR
- Context precision and context recall
- Faithfulness proxy from citation support checks
- Hallucination detection from unsupported cited sentences
- Answer relevancy via local embedding similarity
- Latency benchmarking per question and aggregate
- Ground-truth comparison against reference answers and expected sources/pages

For a real bid-intelligence workflow, keep the evaluation set in version control and run it after every ingestion, model, reranker, or prompt change.

In [111]:
# @title Automated local evaluation: retrieval, generation, hallucination, latency
@dataclass
class GroundTruthCase:
    query: str
    reference_answer: str = ""
    relevant_sources: List[str] = field(default_factory=list)
    relevant_terms: List[str] = field(default_factory=list)


class RAGEvaluator:
    def __init__(self, rag: EnterpriseESGRAG):
        self.rag = rag

    def run(self, cases: Sequence[GroundTruthCase], k: Optional[int] = None) -> pd.DataFrame:
        rows = []
        for case in tqdm(cases, desc="Evaluating"):
            start = now_ms()
            result = self.rag.ask(case.query, use_memory=False)
            elapsed = now_ms() - start
            citations = result["citations"][: k or len(result["citations"])]
            rows.append(self._score_case(case, result, citations, elapsed))
        df = pd.DataFrame(rows)
        audit_logger.log("evaluation", {"cases": len(cases), "summary": df.mean(numeric_only=True).to_dict() if not df.empty else {}})
        return df

    def _score_case(self, case: GroundTruthCase, result: Dict[str, Any], citations: List[Dict[str, Any]], elapsed_ms: int) -> Dict[str, Any]:
        relevance_flags = [self._is_relevant_source(case, src) for src in citations]
        hit_rate = float(any(relevance_flags)) if citations else 0.0
        precision_at_k = float(np.mean(relevance_flags)) if relevance_flags else 0.0

        # recall@k: measure doc coverage (relevant_sources) or term coverage (relevant_terms)
        # separately — conflating term count with expected doc count gives wrong denominators.
        if case.relevant_sources:
            recall_at_k = min(1.0, sum(relevance_flags) / max(1, len(case.relevant_sources)))
        elif case.relevant_terms:
            snippets_blob = " ".join(str(s.get("snippet", "")) for s in citations).lower()
            covered = sum(1 for t in case.relevant_terms if t.lower() in snippets_blob)
            recall_at_k = covered / max(1, len(case.relevant_terms))
        else:
            recall_at_k = 1.0 if citations else 0.0
        mrr = 0.0
        for idx, is_relevant in enumerate(relevance_flags, start=1):
            if is_relevant:
                mrr = 1.0 / idx
                break

        # AP-weighted context_precision (rewards relevant docs ranked higher).
        ap_num, cum_rel = 0.0, 0
        for rank, flag in enumerate(relevance_flags, start=1):
            if flag:
                cum_rel += 1
                ap_num += cum_rel / rank
        context_precision_ap = ap_num / max(1, cum_rel) if cum_rel else 0.0

        # Term-coverage context_recall (fraction of ground-truth terms found in retrieved snippets).
        snippets_blob = " ".join(str(s.get("snippet", "")) for s in citations).lower()
        if case.relevant_terms:
            covered = sum(1 for t in case.relevant_terms if t.lower() in snippets_blob)
            context_recall_term = covered / max(1, len(case.relevant_terms))
        else:
            context_recall_term = recall_at_k  # fallback when no terms defined

        answer = result.get("answer", "")
        answer_relevancy = self._answer_relevancy(case.query, answer, case.reference_answer)
        guard = result.get("guard", {})

        return {
            "query": case.query,
            "hit_rate": hit_rate,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "mrr": mrr,
            "context_precision": context_precision_ap,   # AP-weighted; rewards higher-ranked relevant results
            "context_recall": context_recall_term,           # term-coverage; not a duplicate of recall@k
            "answer_relevancy": answer_relevancy,
            "faithfulness": guard.get("faithfulness", 0.0),
            "hallucination_rate": guard.get("hallucination_rate", 1.0),
            "latency_ms": elapsed_ms,
            "latency_seconds": elapsed_ms / 1000,
            "answer": answer,
            "sources": [src.get("source_id") for src in citations],
        }

    def _is_relevant_source(self, case: GroundTruthCase, source: Dict[str, Any]) -> bool:
        haystack = " ".join(
            [
                str(source.get("source_id", "")),
                str(source.get("doc_name", "")),
                f"page {source.get('page_number', '')}",
                str(source.get("section", "")),
                str(source.get("snippet", "")),
            ]
        ).lower()

        if case.relevant_sources:
            return any(expected.lower() in haystack for expected in case.relevant_sources)
        if case.relevant_terms:
            return any(term.lower() in haystack for term in case.relevant_terms)
        return True

    def _answer_relevancy(self, query: str, answer: str, reference: str = "") -> float:
        target = reference or query
        # Strip extractive preamble and citation labels before embedding.  Without this,
        # the off-topic "Grounded extractive answer because..." prefix dilutes the cosine
        # similarity and suppresses scores to ~0.66-0.69 even for fully on-topic answers.
        clean_ans = re.sub(r"(?i)^grounded extractive answer[^:]*:[\s]*", "", answer.strip())
        clean_ans = re.sub(r"\[S\d+\]", "", clean_ans).strip() or answer
        try:
            vectors = self.rag.text_embedder.encode([clean_ans, target])
            return cosine(vectors[0], vectors[1])
        except Exception:
            answer_terms = set(tokenize(clean_ans))
            target_terms = set(tokenize(target))
            if not target_terms:
                return 0.0
            return len(answer_terms & target_terms) / len(target_terms)


evaluator = RAGEvaluator(rag)

In [112]:
# @title Ground-truth eval set — Deloitte 2025 C-suite Sustainability Report
# Designed to stress-test text, table, and image/chart retrieval separately

EVAL_SET = [

    # ── TEXT retrieval ────────────────────────────────────────────────────────
    # Tests dense passage retrieval from narrative sections
    GroundTruthCase(
        query="What does Deloitte's 2025 survey say about stakeholder pressure to act on sustainability compared to 2022?",
        reference_answer="Across nearly every major stakeholder group fewer respondents feel pressure to act on sustainability than in 2022. Shareholders dropped from 71% to 58%, boards from 75% to 60%, governments from 77% to 58%, customers from 75% to 57%, employees from 65% to 54%.",
        relevant_terms=["stakeholder", "pressure", "2022", "shareholders", "boards", "governments"],
    ),
    GroundTruthCase(
        query="What are the top obstacles C-suite executives cite for deploying sustainability efforts?",
        reference_answer="The top obstacle is difficulty measuring environmental impact (22%), followed by focus on near-term business challenges (21%), lack of sustainable solutions or inputs (21%), and competing operational demands (18%). Cost and lack of policy support were cited by relatively few.",
        relevant_terms=["obstacle", "measuring", "environmental impact", "near-term", "cost"],
    ),
    GroundTruthCase(
        query="According to Deloitte, what is the de facto sustainability roadmap emerging from multiple years of survey data?",
        reference_answer="The roadmap consists of five recurring top actions: implementing technology solutions, using more sustainable materials, developing more sustainable products and services, implementing operational efficiency measures, and tracking and disclosing sustainability metrics.",
        relevant_terms=["roadmap", "technology", "sustainable materials", "operational efficiency", "tracking"],
    ),

    # ── TABLE retrieval ───────────────────────────────────────────────────────
    # Tests structured data extraction — year-over-year comparisons
    GroundTruthCase(
        query="How did the percentage of companies tying senior leaders' compensation to sustainability performance change from 2024 to 2025?",
        reference_answer="It decreased from 43% in 2024 to 36% in 2025.",
        relevant_terms=["compensation", "senior leaders", "43%", "36%", "2024", "2025"],
    ),
    GroundTruthCase(
        query="What percentage of respondents reported decreasing operations emissions by purchasing renewable energy in 2024 versus 2025?",
        reference_answer="49% in 2024 versus 42% in 2025 — a 7 percentage point decline.",
        relevant_terms=["renewable energy", "42%", "49%", "emissions", "purchasing"],
    ),
    GroundTruthCase(
        query="Which sustainability action had the highest percentage of respondents undertaking it in 2025, and what share ranked it highest priority?",
        reference_answer="Implementing technology solutions was undertaken by 46% of respondents, with 22% ranking it their highest priority.",
        relevant_terms=["technology solutions", "46%", "22%", "highest priority", "undertaken"],
    ),

    # ── IMAGE / CHART retrieval ───────────────────────────────────────────────
    # Tests CLIP embedding retrieval of charts and infographics
    GroundTruthCase(
        query="According to the donut chart on business impact of sustainability actions, what percentage of respondents reported a positive impact on revenue generation?",
        reference_answer="66% reported a positive impact on revenue generation, with 31% neutral and 3% negative.",
        relevant_terms=["revenue generation", "66%", "31%", "positive", "donut"],
    ),
    GroundTruthCase(
        query="What does the bar chart on extreme weather personal impact show for extreme heat across 2022 to 2025?",
        reference_answer="Extreme heat personal impact rose from 41% in 2022 and 2023 to 41% in 2024, then declined to 36% in 2025.",
        relevant_terms=["extreme heat", "41%", "36%", "2022", "2025", "weather"],
    ),
    GroundTruthCase(
        query="What percentage of C-suite respondents said they are already using AI for sustainability, according to the survey chart?",
        reference_answer="81% of respondents globally report they are already using AI to further their sustainability efforts, with 16% planning to within the coming year.",
        relevant_terms=["AI", "81%", "16%", "sustainability", "already using"],
    ),

    # ── CROSS-MODAL ───────────────────────────────────────────────────────────
    # Requires combining chart data with surrounding text
    GroundTruthCase(
        query="What share of companies with revenues over $10 billion increased sustainability investments by more than 20%, and how does this compare to the overall survey average?",
        reference_answer="More than one-fifth (22%) of the largest companies with revenues of $10 billion or more increased investments by more than 20%, compared to 14% across all respondents.",
        relevant_terms=["10 billion", "22%", "14%", "investments", "increased significantly"],
    ),
    GroundTruthCase(
        query="What are the top two challenges companies face in executing sustainability reporting requirements, and what percentage cited each?",
        reference_answer="The evolving policy and regulatory environment was cited by 41% of respondents, and data challenges such as lack of data or perceived accuracy were cited by 40%.",
        relevant_terms=["reporting", "regulatory", "41%", "data challenges", "40%"],
    ),
]

if docstore.count_chunks() == 0:
    print("Index is empty. Run ingestion before evaluation.")
else:
    eval_df = evaluator.run(EVAL_SET, k=5)
    display(eval_df)
    print("\nAggregate metrics:")
    display(eval_df.mean(numeric_only=True).to_frame("mean").T)

    # Break down by modality group
    text_rows   = eval_df.iloc[0:3]
    table_rows  = eval_df.iloc[3:6]
    image_rows  = eval_df.iloc[6:9]
    cross_rows  = eval_df.iloc[9:11]

    print("\n── Text retrieval avg ──")
    display(text_rows.mean(numeric_only=True).to_frame("mean").T)
    print("\n── Table retrieval avg ──")
    display(table_rows.mean(numeric_only=True).to_frame("mean").T)
    print("\n── Image/chart retrieval avg ──")
    display(image_rows.mean(numeric_only=True).to_frame("mean").T)
    print("\n── Cross-modal avg ──")
    display(cross_rows.mean(numeric_only=True).to_frame("mean").T)

Evaluating: 100%|██████████| 11/11 [06:08<00:00, 33.50s/it]


,query,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds,answer,sources
0,What does Deloitte's 2025 survey say about sta...,1.0,1.0,0.666667,1.0,1.0,0.666667,0.879738,0.687783,0.5,38748,38.748,Deloitte's 2025 survey indicates that the pres...,[2025-deloitte-global-c-suite-sustainability-r...
1,What are the top obstacles C-suite executives ...,1.0,0.5,1.000000,1.0,1.0,1.000000,0.795404,0.890026,0.0,41625,41.625,The top obstacles for deploying sustainability...,[2025-deloitte-global-c-suite-sustainability-r...
2,"According to Deloitte, what is the de facto su...",1.0,1.0,1.000000,1.0,1.0,1.000000,0.823549,1.000000,0.0,36400,36.400,A de facto roadmap of sustainability actions i...,[2025-deloitte-global-c-suite-sustainability-r...
3,How did the percentage of companies tying seni...,1.0,1.0,1.000000,1.0,1.0,1.000000,0.625106,0.595982,0.0,29406,29.406,The percentage of companies tying senior leade...,[2025-deloitte-global-c-suite-sustainability-r...
4,What percentage of respondents reported decrea...,1.0,1.0,1.000000,1.0,1.0,1.000000,0.470548,0.300000,0.0,22269,22.269,No evidence found for this comparison in the r...,[2025-deloitte-global-c-suite-sustainability-r...
5,Which sustainability action had the highest pe...,1.0,1.0,1.000000,1.0,1.0,1.000000,0.732332,0.741935,0.0,36904,36.904,The sustainability action that had the highest...,[2025-deloitte-global-c-suite-sustainability-r...
6,According to the donut chart on business impac...,1.0,1.0,0.400000,1.0,1.0,0.400000,0.603981,0.845785,0.0,45083,45.083,Ninety percent of respondents reported that re...,[2025-deloitte-global-c-suite-sustainability-r...
7,What does the bar chart on extreme weather per...,1.0,1.0,0.833333,1.0,1.0,0.833333,0.789431,0.713520,0.0,36578,36.578,The bar chart on extreme weather personal impa...,[2025-deloitte-global-c-suite-sustainability-r...
8,What percentage of C-suite respondents said th...,1.0,1.0,0.600000,1.0,1.0,0.600000,0.826000,1.000000,0.0,21843,21.843,Eighty-one percent of respondents globally rep...,[2025-deloitte-global-c-suite-sustainability-r...
9,What share of companies with revenues over $10...,1.0,1.0,0.800000,1.0,1.0,0.800000,0.738176,0.650000,0.0,30069,30.069,The share of companies with revenues over $10 ...,[2025-deloitte-global-c-suite-sustainability-r...



Aggregate metrics:


,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds
mean,1.0,0.954545,0.845455,1.0,1.0,0.845455,0.730151,0.741522,0.045455,33310.454545,33.310455



── Text retrieval avg ──


...

,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds
mean,1.0,1.0,1.0,1.0,1.0,1.0,0.609329,0.545973,0.0,29526.333333,29.526333



── Image/chart retrieval avg ──


,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds
mean,1.0,1.0,0.611111,1.0,1.0,0.611111,0.739804,0.853102,0.0,34501.333333,34.501333



── Cross-modal avg ──


,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds
mean,1.0,1.0,0.9,1.0,1.0,0.9,0.742785,0.690854,0.0,28779.5,28.7795


In [96]:
# @title Ground-truth examples and evaluation run
# Replace or extend these with page-verified labels from your target ESG bid corpus.
EVAL_SET = [
    GroundTruthCase(
        query="What emissions metrics are disclosed?",
        reference_answer="The answer should identify Scope 1, Scope 2, Scope 3 or GHG emissions only when supported by cited pages.",
        relevant_terms=["scope 1", "scope 2", "scope 3", "ghg", "emissions"],
    ),
    GroundTruthCase(
        query="What renewable energy commitments or progress are reported?",
        reference_answer="The answer should cite renewable energy targets, electricity usage, or progress metrics when present.",
        relevant_terms=["renewable", "energy", "electricity"],
    ),
    GroundTruthCase(
        query="What waste or recycling metrics appear in the documents?",
        reference_answer="The answer should cite waste generated, recycled, diverted, hazardous waste, or circular economy evidence.",
        relevant_terms=["waste", "recycling", "recycled", "circular"],
    ),
]

if docstore.count_chunks() == 0:
    print("Index is empty. Run ingestion before evaluation.")
else:
    eval_df = evaluator.run(EVAL_SET, k=5)
    display(eval_df)
    print("\nAggregate metrics:")
    display(eval_df.mean(numeric_only=True).to_frame("mean").T)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 3/3 [02:33<00:00, 51.06s/it]


,query,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds,answer,sources
0,What emissions metrics are disclosed?,1.0,1.0,1.0,1.0,1.0,1.0,0.731816,0.691158,0.0,53393,53.393,The disclosed emissions metrics include Scope ...,[2025-pwc-network-sustainability-report:p48:te...
1,What renewable energy commitments or progress ...,1.0,1.0,1.0,1.0,1.0,1.0,0.675411,0.907594,0.0,59641,59.641,The Nomura Group has oversight of and governs ...,"[Nomura:p73:text:13, 2025-pwc-network-sustaina..."
2,What waste or recycling metrics appear in the ...,1.0,1.0,0.5,1.0,1.0,0.5,0.704620,0.826625,0.5,39104,39.104,The documents mention several waste or recycli...,"[Nomura:p90:text:7, Nomura:p16:text:4]"



Aggregate metrics:


,hit_rate,precision_at_k,recall_at_k,mrr,context_precision,context_recall,answer_relevancy,faithfulness,hallucination_rate,latency_ms,latency_seconds
mean,1.0,1.0,0.833333,1.0,1.0,0.833333,0.703949,0.808459,0.166667,50712.666667,50.712667


## Production Deployment Considerations

For an enterprise bid-document intelligence system, the notebook architecture maps cleanly to services:

- Ingestion service: asynchronous workers for PDF parsing, OCR, table extraction, chart extraction, and local image summaries.
- Storage: object store for PDFs/artifacts, relational DB for metadata/docstore, managed vector DB such as Qdrant, Weaviate, or Chroma server.
- Retrieval API: query rewrite, hybrid retrieval, metadata filters, reranking, context packing, and source lineage.
- Answer API: strict citation prompt, local Ollama model fallback, hallucination guard, answer audit trail, and red-team filters.
- Security: tenant isolation, RBAC, encryption at rest, private networking, PII/contract-term redaction, prompt injection filtering.
- Observability: latency by stage, retrieval hit rate, answer faithfulness, CPU/GPU/RAM telemetry, failed extractor alerts, drift monitoring.
- Evaluation CI: fixed ground-truth benchmark run on every prompt, model, chunking, or retriever change.
- Human review: confidence thresholds route low-faithfulness answers to analyst review before use in bids.

Scalable improvements:

- Move from a 7B local model to a larger quantized model or a governed on-prem inference server.
- Replace local Tesseract with managed OCR/layout services for scanned bid packs.
- Add layout-aware models such as LayoutLM/DocTR for forms and dense tables.
- Use a stronger local vision-language model for chart understanding and capture chart data tables where possible.
- Add metadata filters for issuer, year, region, ESG standard, supplier category, and business unit.
- Use Qdrant/Weaviate hybrid sparse+dense retrieval for server-side scalability.
- Build ESG ontology extraction for commitments, KPIs, baselines, targets, deadlines, owners, and assurance status.
- Add answer comparison workflows for multi-vendor bid responses and compliance matrices.

In [ ]:
# @title Operational helpers: inspect sources, clear memory, and export audit log
def inspect_sources(result: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for source in result.get("citations", []):
        rows.append(
            {
                "label": source.get("label"),
                "doc": source.get("doc_name"),
                "page": source.get("page_number"),
                "type": source.get("modality"),
                "section": source.get("section"),
                "score": source.get("score"),
                "source_id": source.get("source_id"),
                "artifact": source.get("image_path") or source.get("table_csv_path") or source.get("source_path"),
                "snippet": source.get("snippet"),
            }
        )
    return pd.DataFrame(rows)


def clear_conversation_memory() -> None:
    rag.memory = ConversationMemory()
    rag.retriever = HybridRetriever(rag.cfg, rag.vector_store, rag.bm25, rag.docstore)
    print("Conversation memory cleared.")


def read_audit_log(limit: int = 20) -> pd.DataFrame:
    if not config.audit_log_path.exists():
        return pd.DataFrame()
    rows = []
    with open(config.audit_log_path, "r", encoding="utf-8") as handle:
        for line in handle:
            rows.append(json.loads(line))
    return pd.DataFrame(rows[-limit:])


# Example after running a query:
# inspect_sources(sample_results[0])
# read_audit_log()